# Video → npy TimeSformer → ép thành 8576 → so độ giống với npy Baidu (Kaggle, v4.3)

Notebook chạy trên **Kaggle**. Các bước:
1. Đọc video 224p từ Kaggle Dataset.
2. Trích feature **TimeSformer 768 chiều** bằng model Baidu đã **fine-tune trên SoccerNet**. Code là bản PyTorch của model PaddlePaddle gốc, sai lệch < 1e-5.
3. **Học phép ép 768 → 8576**, rồi **đo độ giống với npy Baidu gốc** trên các trận test. Các trận test không được dùng để học.

Không train hay chấm caption. Xem `CHANGELOG.md` trên GitHub để biết thay đổi của từng phiên bản.

## Chuẩn bị (làm 1 lần)

**1. Tạo Kaggle Dataset chứa video và weights.**
- Vào kaggle.com → **Datasets → New Dataset**.
- Upload `Dataset.zip` (video `1_224p.mkv` / `2_224p.mkv`) và file weights `.pdparams` của TimeSformer.
- File weights tải bằng trình duyệt từ link OneDrive trong README của [baidu-research/Soccernet-features](https://github.com/baidu-research/Soccernet-features).
- Kaggle tự giải nén file zip, notebook xử lý được cả hai trường hợp (đã giải nén hoặc còn zip).
- Đặt chế độ **Private**, vì video SoccerNet thuộc NDA.

**2. Trong notebook, cột bên phải:**
- **Add Input:** chọn dataset vừa tạo.
- **Settings → Accelerator:** chọn **GPU T4 x2**. Không chọn P100: PyTorch mới không còn hỗ trợ tốt GPU này.
- **Settings → Internet:** bật **On**, để clone code từ GitHub và tải npy Baidu từ HuggingFace. Tài khoản Kaggle phải xác minh số điện thoại mới bật được Internet.

**3.** Bấm **Run All**, hoặc chạy lần lượt từng cell.

**Kết quả** nằm trong `/kaggle/working/outputs/` (tab **Output**):
- `timesformer768/`: npy 768 chiều của từng hiệp.
- `mapper_timesformer.pt`: phép ép đã học.
- `mapper_timesformer_report.json`: bảng độ giống.

Muốn chạy tiếp ở phiên sau mà không trích lại, thêm output của lần chạy trước làm Input: notebook tự chép lại các npy đã có.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Cấu hình

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"
GIT_BRANCH = "main"
WORK_DIR = "/kaggle/working/soccernet-caption-deltalab"   # code
DATA = "/kaggle/tmp/caption-2024"                         # npy Baidu + nhãn tải từ HuggingFace (không lưu vào Output)
VIDEO_TMP = "/kaggle/tmp/videos_raw"                      # nơi giải nén nếu dataset còn là file .zip
OUT = "/kaggle/working/outputs"                           # kết quả giữ lại trong tab Output
TS_BATCH = 32                                             # T4: 32 (fp16). Nếu hết VRAM thì giảm xuống 16

## 3. Lấy code từ GitHub và cài thư viện

In [ ]:
import os
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1
!grep -m1 "^## v" CHANGELOG.md
!pip install -q SoccerNet av

## 4. Tìm video và weights trong Input, tải npy Baidu gốc

Cell này tìm `*_224p.mkv` và `*.pdparams` trong `/kaggle/input`. Nếu dataset chỉ có file `.zip` thì giải nén vào `/kaggle/tmp`.

In [ ]:
import glob, zipfile, shutil
from huggingface_hub import hf_hub_download
from SoccerNet.utils import getListGames

def find_videos(root):
    return sorted(p for p in glob.glob(f"{root}/**/[12]_224p.mkv", recursive=True) if "__MACOSX" not in p)

videos = find_videos("/kaggle/input")
if not videos:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        print("giải nén", z)
        with zipfile.ZipFile(z) as f:
            f.extractall(VIDEO_TMP, [m for m in f.namelist() if m.endswith("_224p.mkv") and "__MACOSX" not in m])
    videos = find_videos(VIDEO_TMP)
assert videos, "Không thấy video *_224p.mkv: kiểm tra đã Add Input dataset chưa"

# thư mục gốc chứa <giải>/<mùa>/<trận>/1_224p.mkv
VIDEO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(videos[0]))))
VIDEO_GAMES = sorted({os.path.relpath(os.path.dirname(v), VIDEO_ROOT) for v in videos})
print(len(videos), "video,", len(VIDEO_GAMES), "trận, VIDEO_ROOT =", VIDEO_ROOT)

weights = sorted(glob.glob("/kaggle/input/**/*.pdparams", recursive=True))
assert weights, "Không thấy file .pdparams: upload weights TimeSformer vào dataset"
TS_WEIGHTS = weights[0]
print("weights:", TS_WEIGHTS)

for g in VIDEO_GAMES:
    for f, repo, rev in [("Labels-caption.json", "SoccerNet/SN-Labels", "main"),
                         ("1_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings"),
                         ("2_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings")]:
        if not os.path.exists(f"{DATA}/{g}/{f}"):
            hf_hub_download(repo, f"{g}/{f}", repo_type="dataset", revision=rev, local_dir=DATA)

split_of = {g: s for s in ["train", "valid", "test", "challenge"] for g in getListGames([s], task="caption")}
for g in VIDEO_GAMES:
    print(f"{split_of.get(g, '?'):6} {g}")

## 5. Video → npy TimeSformer 768 chiều

- Mỗi hiệp cho ra `1_timesformer768.npy` / `2_timesformer768.npy`, mỗi giây một dòng, cùng số dòng với npy Baidu.
- Các trận được chia cho **2 GPU T4 chạy song song**. Ước tính khoảng 5–10 phút mỗi hiệp trên một T4.
- Log của từng GPU nằm ở `OUT/extract_gpu*.log`.
- npy đã trích được chép vào `OUT/timesformer768/`. Nếu output của lần chạy trước được thêm làm Input, các npy đó được chép lại và **không trích lại**.

In [ ]:
import subprocess, torch, time

def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

# npy của lần chạy trước (output được thêm làm Input)
for prev in glob.glob("/kaggle/input/**/timesformer768", recursive=True):
    sync(prev, DATA)

todo = [g for g in VIDEO_GAMES if not all(os.path.exists(f"{DATA}/{g}/{h}_timesformer768.npy") for h in (1, 2))]
n_gpu = max(torch.cuda.device_count(), 1)
print(f"{len(todo)} trận cần trích, {n_gpu} GPU")
os.makedirs(OUT, exist_ok=True)
procs = []
for k in range(n_gpu):
    games = todo[k::n_gpu]
    if not games:
        continue
    cmd = ["python", "timesformer_extract.py", "--weights", TS_WEIGHTS, "--video_root", VIDEO_ROOT,
           "--out_root", DATA, "--batch_size", str(TS_BATCH), "--games", *games]
    log = open(f"{OUT}/extract_gpu{k}.log", "w")
    procs.append((subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                   env={**os.environ, "CUDA_VISIBLE_DEVICES": str(k)}), log, k))
t0 = time.time()
while any(p.poll() is None for p, _, _ in procs):
    time.sleep(60)
    done = len(glob.glob(f"{DATA}/**/*_timesformer768.npy", recursive=True))
    print(f"{(time.time() - t0) / 60:5.0f} phút: {done}/{2 * len(VIDEO_GAMES)} hiệp xong", flush=True)
for p, log, k in procs:
    log.close()
    print(f"--- GPU {k} (exit {p.returncode})")
    !tail -n 25 {OUT}/extract_gpu{k}.log
sync(DATA, f"{OUT}/timesformer768")
print(len(glob.glob(f"{OUT}/timesformer768/**/*.npy", recursive=True)), "file npy trong", f"{OUT}/timesformer768")

## 6. Ép 768 → 8576 và đo độ giống với npy Baidu gốc

- **Học** phép ép trên các trận **train** (ridge regression, 5 giây TimeSformer quanh giây t).
- **Chọn tham số** trên các trận **valid**.
- **Đo** trên các trận **test**, những trận không được dùng để học.

| Chỉ số | Ý nghĩa |
|---|---|
| `cosine` | Độ giống trung bình với npy Baidu gốc, 1 = giống hệt |
| `cosine_ge_0.95` | Tỉ lệ số giây đạt cosine ≥ 0,95 |
| `cosine_TPN`… | Độ giống theo từng khối trong 5 khối của Baidu |
| `pearson_r_mean`, `R2` | Mức ăn khớp theo từng chiều |
| `baidu_next_second` | Mốc so sánh: Baidu giây t so với giây t+1 của chính nó (khoảng 0,94) |
| `baidu_random_second` | Mốc so sánh: hai giây ngẫu nhiên không liên quan (khoảng 0,67) |

Thêm `--mlp` vào lệnh để thử thêm một mạng MLP nhỏ. Mạng này chỉ được giữ nếu tốt hơn ridge trên valid.

In [ ]:
!python feature_mapper.py train --soccernet_path {DATA} --out {OUT}/mapper_timesformer.pt --train_splits train --val_splits valid --test_splits test

## 7. Bảng kết quả

In [ ]:
import json, pandas as pd
rep = json.load(open(f"{OUT}/mapper_timesformer_report.json"))
print("offset:", rep["offset"], "| context:", rep["context"], "| lambda:", rep["lambda"], "| kind:", rep["kind"])
print("trận:", {k: len(v) for k, v in rep["games"].items()})
table = pd.DataFrame({k: rep[k] for k in ("val", "test") if k in rep})
ref = pd.DataFrame({k.replace("reference_", "mốc_"): rep[k] for k in ("reference_val", "reference_test") if k in rep})
display(table.round(4))
display(ref.round(4))